# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv("../../data/processed/refresh_feature_vector.csv")
print(df.shape)
print(df["is_declining_label"].value_counts(normalize=True))

My lane predicts is_declining_label (trend_direction == "down") — a binary outcome — from page-level signals in the feature vector. Logistic Regression is the right first model because the label is binary, the features are a mix of numeric and categorical signals with no strong reason to expect deep interactions, and I need coefficients I can explain to a non-ML stakeholder ("pages with X and Y tend to decline"). Decision Tree and Random Forest are included as comparisons since they can pick up non-linear thresholds the logistic model would miss, and Random Forest's feature importances double-check that Logistic Regression isn't leaning on something spurious. I did not use Gradient Boosting — with ~30k rows and 52 engineered columns, the extra complexity isn't earning its keep, and the assignment explicitly warns against rewarding complexity alone

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
FEATURES = [c for c in df.columns if c not in
            ["is_declining_label", "client_id", "page_id", "trend_direction"]]
NUMERIC = df[FEATURES].select_dtypes(include=[np.number]).columns.tolist()
CATEGORICAL = [c for c in FEATURES if c not in NUMERIC]

X = df[FEATURES]
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
print(f"train: {len(X_train)}  test: {len(X_test)}  clients overlap: 0")

Split: client-holdout, grouped by client_id.
A random row-level split would leak — pages from the same client share site-wide patterns (template, niche, historical traffic trends), so the model could memorize client quirks rather than learn generalizable signal. The real test of usefulness is: does this work on a client the model has never seen? I use GroupShuffleSplit grouped on client_id so no client appears in both train and test. This matches the Week-4 baseline's split exactly, since the baseline score also has to be evaluated per-client to be a fair comparison.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
])

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return y_true.iloc[order].mean()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8,
                                             class_weight="balanced", random_state=42),
}

results = []
baseline_queue = pd.read_csv("../../data/processed/baseline_refresh_queue.csv")
baseline_test = baseline_queue.loc[baseline_queue["page_id"].isin(df.iloc[test_idx]["page_id"])]
baseline_p50 = precision_at_k(
    y_test.reset_index(drop=True),
    baseline_test.sort_values("page_id")["baseline_score"].reset_index(drop=True), k=50
)
results.append({"model": "Baseline (hand-rule)", "precision@50": baseline_p50})

for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    pipe.fit(X_train, y_train)
    scores = pipe.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "precision@50": precision_at_k(y_test.reset_index(drop=True), scores, k=50),
        "roc_auc": roc_auc_score(y_test, scores),
        "f1": f1_score(y_test, pipe.predict(X_test)),
    })

results_df = pd.DataFrame(results)
results_df

Same client-holdout split, same test set, same metric (Precision@50) as the Week-4 hand-rule baseline. I load the baseline's saved predictions on this test set and score all models on identical ground.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
best_name = results_df.iloc[1:].sort_values("precision@50", ascending=False).iloc[0]["model"]
best_clf = models[best_name]
best_pipe = Pipeline([("prep", preprocess), ("clf", best_clf)])
best_pipe.fit(X_train, y_train)
preds = best_pipe.predict(X_test)
probs = best_pipe.predict_proba(X_test)[:, 1]

errors = X_test.copy()
errors["y_true"] = y_test.values
errors["y_pred"] = preds
errors["score"] = probs
fp = errors[(errors.y_true == 0) & (errors.y_pred == 1)]
fn = errors[(errors.y_true == 1) & (errors.y_pred == 0)]
print(f"False positives: {len(fp)}  False negatives: {len(fn)}")

# feature importance / coefficients
if hasattr(best_clf, "feature_importances_"):
    names = best_pipe.named_steps["prep"].get_feature_names_out()
    imp = pd.Series(best_clf.feature_importances_, index=names).sort_values(ascending=False)
    print(imp.head(10))
elif hasattr(best_clf, "coef_"):
    names = best_pipe.named_steps["prep"].get_feature_names_out()
    coef = pd.Series(best_clf.coef_[0], index=names).sort_values(key=abs, ascending=False)
    print(coef.head(10))

Errors: I inspect false positives (flagged as declining but weren't) and false negatives (missed declines) from the best model, and check whether errors cluster by client size, content age, or category — signals the model may be over- or under-weighting. Feature importance (Random Forest) or coefficients (Logistic Regression) show what the model actually leans on, so I can sanity-check it against domain sense rather than trust the metric blindly.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.